# E20260913115603636696: Sprint 2 - Gemma-2 9B LoRA vs QLoRA

Повторяем frozen E2 protocol с Gemma-2-9B-IT: одинаковые preprocessing и adapter settings для FP16/BF16 LoRA и 4-bit NF4 QLoRA, метрики fold 7 после эпох 1, 2 и 3.

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
for candidate in (PROJECT_ROOT, *PROJECT_ROOT.parents):
    if (candidate / 'configs/project.json').is_file():
        PROJECT_ROOT = candidate
        break
else:
    raise RuntimeError('Open the notebook inside the cloned repository.')

sys.path.insert(0, str(PROJECT_ROOT / 'src'))
from pmldl_llm.gemma2_experiment import run_gemma2_experiment
from pmldl_llm.notebook import load_experiment_setup, run_notebook_experiment

PROJECT_ROOT

## Frozen experiment contract

Config фиксирует официальный Gemma revision, E2 control, folds 0-6 для train, fold 7 для selection и закрытые folds 8/9. Первый запуск обязан быть smoke-test.

In [ ]:
SETUP = load_experiment_setup(
    'configs/experiments/E20260913115603636696.json',
    project_root=PROJECT_ROOT,
)
SETUP

## Train and evaluate

Runner делает GPU/memory preflight, обучает допустимые PEFT arms, пишет per-epoch metrics, private adapters, manifests и fold-7 probabilities, затем закрывает ClearML task.

In [ ]:
def train_and_evaluate(run):
    return run_gemma2_experiment(run, SETUP, PROJECT_ROOT)

In [ ]:
RESULT = run_notebook_experiment(
    train_and_evaluate,
    SETUP,
    project_root=PROJECT_ROOT,
)
RESULT

## Acceptance gates

После smoke-run импортируем output, выполняем `make prepare-full EXPERIMENT=E20260913115603636696` и запускаем clean full run. Кандидат выбирается только на fold 7; folds 8 и 9 этим notebook не открываются.